<a href="https://colab.research.google.com/github/Leon-web-net/Learning_ML/blob/main/Regression/NLP_logistic_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
cd drive/MyDrive/

/content/drive/MyDrive


In [ ]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 9.2 MB/s eta 0:00:00


In [ ]:
# import zipfile
# import os

# zip_path = "NLP_disaster.zip"
# dest_path= "datasets"

# os.makedirs(dest_path, exist_ok=True)

# with zipfile.ZipFile(zip_path, 'r') as f:
#     f.extractall(dest_path)

In [ ]:
import pandas as pd

path = "./datasets/NLP_disaster/train.csv"
df = pd.read_csv(path)
print(df.shape)
df.head()

(7613, 5)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [ ]:
summary = pd.DataFrame({
    "unique_count": df.nunique(dropna=True),
    "unique_count_per": (df.nunique(dropna=True) / len(df) * 100).round(2),
    "nan_count": df.isna().sum(),
    "nan_count_per": (df.isna().sum() / len(df) * 100).round(2),
    "Length": len(df)
})

summary.T

,id,keyword,location,text,target
unique_count,7613.0,221.0,3341.00,7503.00,2.00
unique_count_per,100.0,2.9,43.89,98.56,0.03
nan_count,0.0,61.0,2533.00,0.00,0.00
nan_count_per,0.0,0.8,33.27,0.00,0.00
Length,7613.0,7613.0,7613.00,7613.00,7613.00


In [ ]:
import pandas as pd
import numpy as np
import random
import re, emoji
from sklearn.model_selection import StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, precision_recall_curve, confusion_matrix

SEED = 42
np.random.seed(SEED)
random.seed(SEED)

train_path = "./datasets/NLP_disaster/train.csv"

df = pd.read_csv(train_path)

df.drop(columns=["location"], inplace=True)
df["keyword"]= df["keyword"].fillna("misssing").str.lower().str.strip()
df.head()


,id,keyword,text,target
0,1,misssing,Our Deeds are the Reason of this #earthquake M...,1
1,4,misssing,Forest fire near La Ronge Sask. Canada,1
2,5,misssing,All residents asked to 'shelter in place' are ...,1
3,6,misssing,"13,000 people receive #wildfires evacuation or...",1
4,7,misssing,Just got sent this photo from Ruby #Alaska as ...,1


## Text cleaning functions

- `replace_emoji`: helper function that replaces any emoji in text with a placeholder token (default `" <emoji> "`).  

- `basic_clean`:  
  - Lowercases text.  
  - Replaces URLs with `<url>`.  
  - Replaces `@username` with `<user>`.  
  - Strips `#` from hashtags but keeps the word.  
  - Replaces emojis with `<emoji>`.  
  - Normalises whitespace.  
  - Returns a clean, simplified version of the input.  

- `raw_lite_clean`:  
  - A lighter version: lowercase, replace URLs, user mentions, and emojis.  
  - Does not handle hashtags.  
  - Normalises whitespace.  

These functions standardise raw text for NLP models by reducing noise and ensuring consistent tokens.  


In [ ]:
def replace_emoji(text:str, repl=" <emoji> "):
    return emoji.replace_emoji(text,replace=repl)

def basic_clean(s:str)->str:
    s = str(s).lower()
    s = re.sub(r"http\S+|www\.\S+","<url> ",s)
    s = re.sub(r"@\w+", " <user> ",s)
    s = re.sub(r"#(\w+)",r"\1",s)
    s = emoji.replace_emoji(s,replace=" <emoji> ")
    s = re.sub(r"\s+"," ",s).strip()
    return s

def raw_lite_clean(s:str)->str:

    s = str(s).lower()
    s = re.sub(r"http\S+|www\.\S+", " <url> ", s)
    s = re.sub(r"@\w+", " <user> ", s)
    s = replace_emoji(s, repl=" <emoji> ")
    s = re.sub(r"\s+", " ", s).strip()

    return s

In [ ]:
df["text_raw"] = df["text"].astype(str)

df["text_clean"] = df["text_raw"].map(basic_clean)
df["text_raw_lite"] = df["text_raw"].map(raw_lite_clean)

In [ ]:
def add_simple_feats(df:pd.DataFrame) -> pd.DataFrame:
    t = df["text_clean"]
    df["has_url"] = t.str.contains("<url>").astype(int)
    df["has_user"] = t.str.contains("<user>").astype(int)
    df["has_emoji"] =t.str.contains("<emoji").astype(int)
    df["text_len"] = df["text_raw"].str.len().fillna(0)
    df["word_len"] = df["text_raw"].str.split().map(len).fillna(0)
    df["digits_ratio"] = df["text_raw"].str.count(r"\d").div(df["text_len"].replace(0,1))
    df["punct_ratio"] = df["text_raw"].str.count(r"[^\w\s]").div(df["text_len"].replace(0,1))
    return df


In [ ]:
df = add_simple_feats(df)

num_cols = ["text_len","word_len",
            "digits_ratio","punct_ratio",
            "has_url","has_user","has_emoji"
           ]
df.head()

,id,keyword,text,target,text_raw,text_clean,text_raw_lite,has_url,has_user,has_emoji,text_len,word_len,digits_ratio,punct_ratio
0,1,misssing,Our Deeds are the Reason of this #earthquake M...,1,Our Deeds are the Reason of this #earthquake M...,our deeds are the reason of this earthquake ma...,our deeds are the reason of this #earthquake m...,0,0,0,69,13,0.000000,0.014493
1,4,misssing,Forest fire near La Ronge Sask. Canada,1,Forest fire near La Ronge Sask. Canada,forest fire near la ronge sask. canada,forest fire near la ronge sask. canada,0,0,0,38,7,0.000000,0.026316
2,5,misssing,All residents asked to 'shelter in place' are ...,1,All residents asked to 'shelter in place' are ...,all residents asked to 'shelter in place' are ...,all residents asked to 'shelter in place' are ...,0,0,0,133,22,0.000000,0.022556
3,6,misssing,"13,000 people receive #wildfires evacuation or...",1,"13,000 people receive #wildfires evacuation or...","13,000 people receive wildfires evacuation ord...","13,000 people receive #wildfires evacuation or...",0,0,0,65,8,0.076923,0.030769
4,7,misssing,Just got sent this photo from Ruby #Alaska as ...,1,Just got sent this photo from Ruby #Alaska as ...,just got sent this photo from ruby alaska as s...,just got sent this photo from ruby #alaska as ...,0,0,0,88,16,0.000000,0.022727


In [ ]:
word_vect = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=3,
    max_features=None,
    strip_accents="unicode",
    sublinear_tf=True
)

char_vect = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    min_df=3,
    sublinear_tf=True,
)

In [ ]:
kw_ohe = OneHotEncoder(handle_unknown="ignore",sparse_output=True)
num_scaler = StandardScaler(with_mean=True)

features = ColumnTransformer(
    transformers=[
        ("word",word_vect,"text_clean"),
        ("char",char_vect,"text_raw_lite"),
        ("kw",kw_ohe,["keyword"]),
        ("num",num_scaler,num_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3, # if kept as sparse or converted to dense
    n_jobs=None # CPU cores for parallel compute
)

## Logistic regression with L2 penalty (pipeline context)

You are fitting a logistic regression with L2 regularisation via `liblinear` inside a `Pipeline`.

### Objective

$$
L(w)=\sum_{i=1}^{N}\log\!\big(1+\exp(-y_i\,w^\top x_i)\big)\;+\;\frac{\lambda}{2}\,\lVert w\rVert_2^{2},
\qquad y_i\in\{-1,+1\}.
$$

- First term = logistic loss (data fit).  
- Second term = L2 penalty that shrinks coefficients.

### Properties

- $L(w)$ is convex in $w$.  
- Larger C ⇒ weaker regularisation (potentially lower bias, higher variance).  
- Smaller C ⇒ stronger shrinkage (higher bias, lower variance).


In [ ]:
clf = LogisticRegression(
    solver="liblinear",
    penalty="l2",
    C= 1.0,
    max_iter=2000,
    random_state= SEED,
)

pipe = Pipeline([
    ("features",features),
    ("clf",clf)
])

## Stratified K-Fold cross-validation with logistic regression pipeline

### Setup
- Use `StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)` to split the data.  
- This preserves class balance across folds.  

### Process
For each fold:
- Train the pipeline on training folds.  
- Predict probabilities on the validation fold.  
- Store out-of-fold predictions in `oof_pred`.  
- Apply a default threshold of 0.5 to convert probabilities into class labels.  
- Compute the F1 score on that fold and record it.  

### Output
- Print the F1 score for each fold.  
- Report the mean and standard deviation across folds:  

$$
\text{CV F1@0.50} = \frac{1}{K} \sum_{k=1}^{K} F1_k,
\quad \text{with spread measured by } \sigma(F1_k).
$$

### Purpose
- Provides an unbiased estimate of model performance on unseen data.  
- `oof_pred` can be saved for meta-learning or calibration.  
- Variability across folds highlights sensitivity to train/validation splits.


In [ ]:
skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=SEED)

y = df["target"].astype(int).values
oof_pred = np.zeros(len(df),dtype=float)

fold_scores = []
for fold, (tr_idx, va_idx) in enumerate(skf.split(df, y), 1):
    tr_df = df.iloc[tr_idx]
    va_df = df.iloc[va_idx]
    y_tr = y[tr_idx]
    y_va = y[va_idx]

    model = pipe
    model.fit(tr_df, y_tr)
    proba = model.predict_proba(va_df)[:,1]
    oof_pred[va_idx] = proba

    # quick default threshold 0.5
    preds = (proba >= 0.5).astype(int)
    f1 = f1_score(y_va, preds)
    fold_scores.append(f1)
    print(f"Fold {fold}: F1@0.50 = {f1:.4f}")

print(f"CV F1@0.50: mean={np.mean(fold_scores):.4f}  std={np.std(fold_scores):.4f}")

Fold 1: F1@0.50 = 0.7925
Fold 2: F1@0.50 = 0.7599
Fold 3: F1@0.50 = 0.7508
Fold 4: F1@0.50 = 0.7543
Fold 5: F1@0.50 = 0.7790
CV F1@0.50: mean=0.7673  std=0.0159


## Threshold tuning on OOF predictions

### Grid search
- Define a grid of thresholds from 0.30 to 0.70 (step 0.01).  

- Keep track of the threshold giving the best F1.

### Result
- Print the best threshold and corresponding OOF F1 score.  

$$
t^{*} = \arg\max_{t \in \text{grid}} F1(y,\; \mathbf{1}[oof\_pred \geq t])
$$

### Diagnostics
- Compute the confusion matrix at the chosen threshold.  
- This shows the balance between true positives, false positives, true negatives, and false negatives.  

### Purpose
- Default threshold 0.5 is not always optimal.  
- Tuning on OOF predictions finds a threshold that maximises the evaluation metric.  


In [ ]:
thr_grid = np.round(np.arange(0.30, 0.701, 0.01), 2)
best_thr, best_f1 = 0.50, -1.0
for t in thr_grid:
    f1 = f1_score(y, (oof_pred >= t).astype(int))
    if f1 > best_f1:
        best_f1, best_thr = f1, t

print(f"Best OOF threshold = {best_thr:.2f}, OOF F1 = {best_f1:.4f}")

# Optional diagnostics
cm = confusion_matrix(y, (oof_pred >= best_thr).astype(int))
print("OOF Confusion Matrix @ best thr:")
print(cm)

Best OOF threshold = 0.46, OOF F1 = 0.7723
OOF Confusion Matrix @ best thr:
[[3693  649]
 [ 805 2466]]
